Using this to test functions - compare betas from that pipeline to results from this.

Use just the beta images and the summed images.

Test with all data and missing data.

Note that it doesn't matter what voxel we use; for each subject-week, all their voxel values are exactly the same (in simulations).

In [1]:
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn import plotting as npl
import os
import statsmodels.formula.api as smf
import smarts_cerebellum.globals as gl
from smarts_cerebellum.util import subj_path_search

In [2]:
all_weeks = [0,4,12,24,52]

subj = 'subj0'
week = 0


df = pd.read_csv(f'{gl.baseDir}/simulations/participants_sim.tsv', sep = '\t')

dictionaries = []
for week in all_weeks:
    search_path = f'{gl.baseDir}/simulations/{subj}/{subj}_W{week}_simulated.nii.gz'
    paths, subjs = subj_path_search(search_path, subj, week, df = df)
    dictionaries.append(dict(zip(subjs, paths)))

In [3]:
y_vals = []
week_order = [0, 4, 12, 24, 52]

for w, week in enumerate(all_weeks):
    for subj in df.subj_id.unique():
        img = nib.load(dictionaries[w][f'{subj}'])
        arr = img.get_fdata()
        voxel = arr[1,1,1]
        y_vals.append({'subj_id': subj, 'week': week, 'y': voxel})

y_df = pd.DataFrame.from_records(y_vals, columns = ['subj_id', 'week', 'y']) # try using from_records instead of from_dict?
y_df['week'] = pd.Categorical(y_df['week'], categories = week_order, ordered = True)

A potential issue: in `model.summary()`, it has the weeks written out of order, so W0 (intercept), W12, W24, W4, W52, instead of W4, ..

Fix: use pd categorical ordering (when making the dataframe).

Test with this fix in the function `lme.py` and see if it changes things.

NO...this is not an issue, since we use int for Weeks (when passing it), i think....

Maybe just have that line in there in case?

In [4]:
model = smf.mixedlm('y ~ C(week)', data=y_df, groups='subj_id').fit()
model.summary()

/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/.venv/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/.venv/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


<class 'statsmodels.iolib.summary2.Summary'>
"""
         Mixed Linear Model Regression Results
========================================================
Model:              MixedLM Dependent Variable: y       
No. Observations:   50      Method:             REML    
No. Groups:         10      Scale:              0.0899  
Min. group size:    5       Log-Likelihood:     -15.3930
Max. group size:    5       Converged:          Yes     
Mean group size:    5.0                                 
--------------------------------------------------------
              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      0.698    0.095  7.363 0.000  0.512  0.884
C(week)[T.4]  -0.371    0.134 -2.765 0.006 -0.633 -0.108
C(week)[T.12] -0.222    0.134 -1.656 0.098 -0.485  0.041
C(week)[T.24] -0.221    0.134 -1.646 0.100 -0.483  0.042
C(week)[T.52] -0.231    0.134 -1.722 0.085 -0.494  0.032
subj_id Var    0.000    0.050                           
========================================================

"""

In [5]:
# beta, se images from pipeline

lmes_path = os.path.join(gl.baseDir, 'simulations', 'lme_images')

for week in all_weeks:
    print(f"beta, se for week {week}")

    beta_img = nib.load(f'{lmes_path}/sims_lme_W{week}_lme_beta.nii.gz')
    beta_arr = beta_img.get_fdata()

    print(np.round(beta_arr[1,1,1], 3))

    bse_img = nib.load(f'{lmes_path}/sims_lme_W{week}_lme_bse.nii.gz')
    bse_arr = bse_img.get_fdata()

    print(np.round(bse_arr[1,1,1], 3))


beta, se for week 0
0.698
0.095
beta, se for week 4
-0.371
0.134
beta, se for week 12
-0.222
0.134
beta, se for week 24
-0.221
0.134
beta, se for week 52
-0.231
0.134
